# 걸스라미 · 구글 공식 사진 피팅 1장 시험

일반 CPU 런타임에서 실행합니다. GPU 대여는 필요하지 않습니다. Google Cloud 프로젝트에 결제와 `aiplatform.googleapis.com` 사용이 설정되어 있어야 합니다. 이 노트북은 프로젝트·결제·API 설정을 변경하지 않습니다.

2026-10-01 확인 기준 결과 한 장은 USD $0.06입니다. 세금·결제 통화는 계정 요금에 따릅니다. 한 세션에서 한 번만 생성 요청하고, 실패해도 재시도하지 않습니다. 인증 셀을 다시 실행하거나 새 런타임을 만들면 횟수 제한도 새로 시작합니다. 계정 전체의 결제 한도가 아닙니다.

구글 도쿄 `asia-northeast1`의 `virtual-try-on-001`을 사용합니다. 권리가 확인된 비개인 시험 이미지로 먼저 확인하세요. 본인 사진은 Google Cloud로의 전송·처리에 동의한 경우에만 사용하세요. Drive·Cloud Storage에 사진을 저장하지 않습니다. 노트북의 출력은 공유할 때 함께 보일 수 있으니 공유 전 출력을 지우세요.

우리 저장소의 24시간 제한은 구글의 모든 처리 기록 삭제를 보장하지 않습니다. 일반 고객 사진은 처리 지역·제공사·보관 조건을 안내하고 별도 동의를 확인한 후에만 연결해야 합니다. 이 시험 노트북은 고객 작업 큐를 열거나 사이트 사진 접수를 켜지 않습니다.

[API](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/capabilities/generate-virtual-try-on-images?hl=en) · [가격](https://cloud.google.com/gemini-enterprise-agent-platform/generative-ai/pricing) · [데이터 처리 조건](https://docs.cloud.google.com/gemini-enterprise-agent-platform/resources/zero-data-retention)


In [ ]:
%pip install pillow==12.3.0 "google-auth[requests]==2.59.1"


In [ ]:
import base64
import binascii
import io
import json
import re
from urllib.error import HTTPError
from urllib.request import HTTPRedirectHandler, Request, build_opener

from PIL import Image, ImageOps, UnidentifiedImageError


class TryOnError(Exception):
    pass


class NoRedirect(HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        raise TryOnError("HTTP_REDIRECT_NOT_ALLOWED")


def _image_format(data):
    if not isinstance(data, bytes) or not data or len(data) > 7_000_000:
        raise TryOnError("IMAGE_SIZE_INVALID")
    try:
        with Image.open(io.BytesIO(data)) as image:
            format = image.format
            if format not in ("JPEG", "PNG") or image.width * image.height > 16_777_216 or max(image.size) > 4096:
                raise TryOnError("IMAGE_FORMAT_INVALID")
            if format == "JPEG" and not data.endswith(b"\xff\xd9"):
                raise TryOnError("IMAGE_INVALID")
            image.verify()
        with Image.open(io.BytesIO(data)) as decoded:
            decoded.load()
        return format
    except (UnidentifiedImageError, OSError, ValueError, Image.DecompressionBombError):
        raise TryOnError("IMAGE_INVALID") from None


def private_jpeg(data):
    _image_format(data)
    with Image.open(io.BytesIO(data)) as source:
        oriented = ImageOps.exif_transpose(source)
        if oriented.mode in ("RGBA", "LA") or "transparency" in oriented.info:
            rgba = oriented.convert("RGBA")
            image = Image.new("RGB", rgba.size, "white")
            image.paste(rgba, mask=rgba.getchannel("A"))
        else:
            image = oriented.convert("RGB")
        image.thumbnail((1536, 1536))
        output = io.BytesIO()
        image.save(output, format="JPEG", quality=92)
        return output.getvalue()


class GoogleTryOnClient:
    def __init__(self, project, access_token, *, max_requests, approved=False, opener=None):
        if not isinstance(project, str) or not re.fullmatch(r"[a-z][a-z0-9-]{4,28}[a-z0-9]", project):
            raise TryOnError("PROJECT_ID_INVALID")
        if type(max_requests) is not int or not 1 <= max_requests <= 10:
            raise TryOnError("REQUEST_LIMIT_INVALID")
        if not callable(access_token):
            raise TryOnError("GOOGLE_AUTH_REQUIRED")
        self._endpoint = "https://asia-northeast1-aiplatform.googleapis.com/v1/projects/" + project + "/locations/asia-northeast1/publishers/google/models/virtual-try-on-001:predict"
        self._access_token = access_token
        self._remaining = max_requests
        self._approved = approved is True
        self._open = opener or build_opener(NoRedirect()).open

    @property
    def remaining(self):
        return self._remaining

    def generate(self, person, garment):
        if not self._approved:
            raise TryOnError("PAID_RUN_NOT_APPROVED")
        if self._remaining == 0:
            raise TryOnError("REQUEST_LIMIT_REACHED")
        _image_format(person)
        _image_format(garment)
        try:
            credential = self._access_token()
        except Exception:
            raise TryOnError("GOOGLE_AUTH_FAILED") from None
        if not isinstance(credential, str) or not credential or any(c.isspace() for c in credential):
            raise TryOnError("GOOGLE_AUTH_REQUIRED")
        payload = {
            "instances": [{
                "personImage": {"image": {"bytesBase64Encoded": base64.b64encode(person).decode()}},
                "productImages": [{"image": {"bytesBase64Encoded": base64.b64encode(garment).decode()}}],
            }],
            "parameters": {"sampleCount": 1},
        }
        request = Request(self._endpoint, data=json.dumps(payload).encode(), method="POST", headers={"Content-Type": "application/json", "Authorization": "Bearer " + credential})
        self._remaining -= 1
        try:
            with self._open(request, timeout=120) as response:
                data = response.read(10_000_001)
            if len(data) > 10_000_000:
                raise TryOnError("GOOGLE_RESPONSE_TOO_LARGE")
            result = json.loads(data)
            predictions = result.get("predictions") if isinstance(result, dict) else None
            if not isinstance(predictions, list) or len(predictions) != 1 or not isinstance(predictions[0], dict):
                raise TryOnError("GOOGLE_RESULT_INVALID")
            prediction = predictions[0]
            if prediction.get("raiFilteredReason"):
                raise TryOnError("IMAGE_FILTERED")
            encoded = prediction.get("bytesBase64Encoded")
            if not isinstance(encoded, str) or len(encoded) > 9_333_336:
                raise TryOnError("GOOGLE_RESULT_INVALID")
            image = base64.b64decode(encoded, validate=True)
            format = _image_format(image)
            if prediction.get("mimeType") != {"JPEG": "image/jpeg", "PNG": "image/png"}[format]:
                raise TryOnError("GOOGLE_RESULT_INVALID")
            return image
        except HTTPError as error:
            code = error.code
            error.close()
            raise TryOnError("GOOGLE_HTTP_" + str(code)) from None
        except (OSError, ValueError, binascii.Error):
            raise TryOnError("GOOGLE_REQUEST_FAILED") from None


## 인증과 1회 비용 확인
이미 사용하는 Google Cloud 프로젝트 ID를 입력하세요. 새 프로젝트를 만들거나 결제 수단을 등록하지 않습니다. 로그인과 Google 권한 동의는 직접 완료하세요. 비밀번호·API 키·service account 키를 노트북에 붙여 넣지 않습니다.


In [ ]:
from google.colab import auth
import google.auth
from google.auth.transport.requests import Request as AuthRequest

client = credentials = None
project_id = input('사용할 Google Cloud 프로젝트 ID: ').strip()
GoogleTryOnClient(project_id, lambda: '', max_requests=1)
confirmation = input('구글 도쿄 사진 처리와 1회 생성(USD $0.06, 별도 세금·환율) 승인: 승인 을 입력: ').strip()
if confirmation != '승인':
    raise TryOnError('PAID_RUN_NOT_APPROVED')
auth.authenticate_user(project_id=project_id)
credentials, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])

def access_token():
    if not credentials.valid:
        credentials.refresh(AuthRequest())
    return credentials.token

client = GoogleTryOnClient(project_id, access_token, max_requests=1, approved=True)
print('1회 시험 준비 완료. 아직 생성 요청은 보내지 않았습니다.')


## 정면 인물 사진 → 해당 니트 사진
7MB 이하 JPEG/PNG를 한 장씩 선택하세요. 긴 변 4096px·전체 16메가픽셀 이하의 사진을 처리합니다. 사진은 1536px로 줄이고 위치·촬영 정보를 제거합니다. 실제 상품 사진과 동일한 색상을 시험하세요.

결과의 얼굴·체형, 니트 색·무늬·목선, 소매·손을 확인하세요. 생성 이미지는 사이즈나 실제 착용감을 측정하지 않습니다. 다른 색을 시험하려면 그 색의 실제 상품 사진이 필요합니다.


In [ ]:
from google.colab import files
import tempfile
import shutil

def upload_photo(label):
    print(label)
    directory = tempfile.mkdtemp(prefix='girlsrami-trial-')
    try:
        uploads = files.upload(target_dir=directory)
        if len(uploads) != 1:
            raise TryOnError('ONE_IMAGE_REQUIRED')
        return private_jpeg(next(iter(uploads.values())))
    finally:
        shutil.rmtree(directory)

person = garment = result_bytes = None
try:
    if client is None or client.remaining == 0:
        raise TryOnError('REQUEST_LIMIT_REACHED')
    person = upload_photo('정면 인물 사진 1장')
    garment = upload_photo('해당 색상의 실제 니트 사진 1장')
    result_bytes = client.generate(person, garment)
    display(Image.open(io.BytesIO(result_bytes)))
    print('생성 결과를 직접 확인하세요. 남은 요청:', client.remaining)
except TryOnError as error:
    print('시험 중지:', str(error))
finally:
    person = garment = result_bytes = None


## 시험 종료
개인 사진을 사용했다면 노트북 메뉴에서 모든 출력을 지운 후 런타임을 해제하세요. 이 작업은 Google 제공사의 처리 기록 삭제 요청을 대신하지 않습니다.


In [ ]:
from google.colab import runtime
client = credentials = None
runtime.unassign()
